# Pycavi zero point calibration check

This notebook checks the calibration of swarped image calibrated in the previous step.

- Reads the swarped image and some necessary keywords.
- Read the catalogue used to calibrate the image.
- Produce several plots that can be used to determine the quality of the photometry.
- Saves the resulting plot into a file

The next step is to determine the illumination correction, an effect that can
imprint systematic variations abobe 0.2 mags accross the field for wide field
imagers such as OmegaCam and WFI. Pycavi offers two ways to do so: 

    1.- chebyshev polynomial fits, fast and convenient because one can save a limited number ofn coefficients. The user should test with different degree polynomials and check the residual standard deviation of each, together with the presence of possible large scale variations. It is suggested to use the lowest degree polynomials that gives the minimum  standard deviation. Experimentation shows that the standard deviation decreases as one increases the degree, until a minimum, and then increases due to polynomial oscilations at larger degrees. Chebyshev gives reasonable solutions with very low degrees of freedom.

    2.- Kriging, not as well known, but very powerful as it permits the best unbiased estimator when we interpolate data between points. The disadvantage of Krigging is that it could usse all stars in the fields to carry out the calculations, depending on the correlations of the data, this last one given by the variogram. For filters such as NB_659 Krigging gives an advantage over Chebyshev as it takes better care of short range correlation. In the case of OmegaCam NB_659, using kriging one *rediscovers the four quadrants of the filter*.

        - When kriging the user needs to determine the best variogram model and can play with the parameters: 
    
        maxlag: maximum distance between two points that will be used in the modeling of the variogram
        nlags:  number of bins between 0 and maxlag, tipically 200 - 500.
        nugget: smaller value of the correlation aty short distances. When the residuals are noisy this number will be larger as it gives the short scale variance of the data (random errors of the photometry).

        - When kriging the first step is to determine the variogram, a 1d function that gives the correlations of the data-values as a function of distance. This function starts at zero with a value called the "nugget", and increases up to a maximum after which should remains flat. Except that it does not due to edge and other effects. The value of distance where the varigram flattens indicate the point at which the data is not correlated anymore. The *nugget* gives the value of the short scale variance of the data. After the varigram is determined we fit one of the approximation functions: gaussian, exponential, or spherical. Afer this we use the UniversalKriging function of the python package pykrige to do the actual determination of covariances which are used for interpolation.

        - The error determination when kriging is delicate because the interpolation is exact for the points used for the interpolation. We offer two possible paths: (1) use the same points used for the kriging interpolation determination, but with a small adjustable offset, dpix. This is the quick way that should be used for a first run until the best kriging parameters are determined. Once the user is happy with the parameters, then the k-fold method can be selected where a (100-k)% of points are used for interpolation, and the remaianing k% are used to determine residuals. This means that roof(100/k) number of iterations will need to be performed.

- Then the illumination correction is shown as a colout image with contours overlaid.
The image is saved to a file with root name equal to the image root name in the
Calibration directory.

- After the illumination correction determination the photometry is corrected and the
residuals are ploted for both the corrected magnitudes and the raw magnitudes. Two plots
are generated for QC purposes: a residuals versus x-coordinates for the uncorrected points
in red, and the corrected points in black; and a histogram of the residuals for both
sets of points. The mean, median and 1-sigma variation is reported in these plots.
The plots are saved with the root-name of the image plus a string giving the type
of correction and the string _res_01.png at the end.

- Another set of plots is presented with the residuals as a function of radius, azimuthal
angle measured from the center of the FoV, and versus x and y. These plots have similar names
as the previous set, but with end string _res_02.png.



In [ ]:
import os
import sys
import glob
import copy
import logging

import numpy as np
from numpy.polynomial import chebyshev
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec

from astropy.stats import sigma_clipped_stats, sigma_clip
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

from scipy.interpolate import griddata, Rbf
from scipy.stats import norm

from photutils.aperture import SkyCircularAperture, SkyCircularAnnulus, aperture_photometry


#   from pycavi_astromatic import SExtractor
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production

from pycavi import pycavi_tools
from pycavi.pycavi_zpcal import extract_calibration, plot_zpcal, get_files_dict
from pycavi.pycavi_chebyshev import fit_chebyshev, chebyshev_2d_eval, chebyshev_polynomials, evaluate_2dpoly, append_hashline
from pycavi.pycavi_kriging import kriging_interpolation, determine_kriging_errors


In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/../config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_zpcal_check.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
filters = []
reject_observations = []
dir_name = config['Directories']['cal_dir_name']
zpcal_dict = get_files_dict(reduced_std_dir, reduced_science_dir, dir_name, filters, reject_observations)

observation_tags = list(set(zpcal_dict.keys()))
observation_tags.sort()    # Needed to ensure same index in every run
images_to_check = []
for i in range(len(observation_tags)):
    logging.debug("index: {} observation_tag: {}".format(i, observation_tags[i]))
    for j in range(len(zpcal_dict[observation_tags[i]])):
        images_to_check.append(zpcal_dict[observation_tags[i]][j])

for i in range(len(images_to_check)):
    logging.info("index: {} -- file: {}".format(i, images_to_check[i]))



In [ ]:
# Here we loop over the images selected
# See list above to select a data partition to calibrate
# For example, if the list is:
#
# index: 0 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/Swarp_dir/CP_Pup_r_SDSS.coadd.fits
# index: 1 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_g_SDSS.coadd.fits
# index: 2 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_r_SDSS.coadd.fits
# index: 3 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_NB_659.coadd.fits
# index: 4 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_g_SDSS.coadd.fits
# index: 5 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_r_SDSS.coadd.fits
# index: 6 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/Swarp_dir/SA98_NB_659.coadd.fits
#
# then to calibrate M67_NB_659.coadd.fits set
#
# idx_selected = [3]
#
idx_selected = [6]    # Keep empty to check all files
for i in range(len(images_to_check)):
    
    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    zpcal = extract_calibration(images_to_check[i])

    image_dir = os.path.dirname(zpcal.image_path)
    image_name = os.path.basename(zpcal.image_path)[:-5]     # Note that it assumes that we strip the fits extension
    image_path = image_dir+'/'+image_name

    # we read ORIGFILE for proper reference in coefficients file
    origfile = zpcal.origfile
    
    plot_zpcal(zpcal, img_scaling=4)
    
logging.info("-------Done-------")

In [ ]:
###################################### 01 FILTER DATA ######################################
#
# Here we characterize the spatial properties of the magnitude residuals before correction


# The data is filtered to get obviously bad values and values
# of faint stars which suffer of intrinsic errors which are
# larger than the effect we want to measure.
#
# The call to the filter_data procedure creates the masked arrays
# zpcal.<array>_masked, where array can be any of x, y, z, etc.
#
n_before_filter = len(zpcal.x)
zpcal.filter_data(dm_min=-0.4, dm_max=0.4, brightest_mag=12.0, faintest_mag=15.5)
n_after_filter = len(zpcal.x_masked)
logging.info("Filtering: number of data points {} ---> {}".format(n_before_filter, n_after_filter))

# We create a ds9 regin file for inspection
zpcal.create_ds9_region()

########## 02  PARTITION DATA: MEASURE AND UNCERTAINTY DETERMINATION PARTITIONS ############
#
#.   subset: to be used to determine approximation
#.   check: to be used to check rms of approximation
#
x_subset = zpcal.x_masked
y_subset = zpcal.y_masked
z_subset = zpcal.z_masked
z_subset_err = zpcal.z_masked_err
dimx = zpcal.naxis1
dimy = zpcal.naxis2
origfile = zpcal.origfile

# some statistics
z_mean_subset, z_median_subset, z_std_subset = sigma_clipped_stats(z_subset, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_subset, 4), round(z_median_subset, 4), round(z_std_subset, 4)))

In [ ]:
############################ 03 PLOT RESIDUALS AND PDF FOR UNCORRECTED DATA ####################
#
fig = plt.figure(figsize=(12,5))

gs = GridSpec(1, 3, figure = fig)

ax1 = fig.add_subplot(gs[0, 0:2])
ax1.scatter(x_subset, z_subset, s=2, color='red')

ax1.axhline(z_mean_subset, color='k', linewidth=1, linestyle='-', label='mean: {}'.format(round(z_mean_subset, 3)))
ax1.axhline(z_mean_subset+z_std_subset, color='k', linewidth=1, linestyle='--', label='sigma: {}'.format(round(z_std_subset, 3)))
ax1.axhline(z_mean_subset-z_std_subset, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.set_ylim(-0.25, 0.25)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

ax2 = fig.add_subplot(gs[0, 2])
z_val = np.linspace(-0.2, 0.2, 100)
p = norm.pdf(z_val, z_mean_subset, z_std_subset)
ax2.hist(z_subset, bins=30, density=True, stacked=True, color='green', alpha=0.7)
plt.plot(z_val, p, 'k', linewidth=2)

ax2.axvline(z_mean_subset, color='k', linewidth=1, linestyle='-', label='mean: {}'.format(round(z_mean_subset, 3)))
ax2.axvline(z_mean_subset+z_std_subset, color='k', linewidth=1, linestyle='--', label='sigma: {}'.format(round(z_std_subset, 3)))
ax2.axvline(z_mean_subset-z_std_subset, color='k', linewidth=1, linestyle='--')
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm [mag]", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Here we fit the data either with Chebyshev or kriging method
# Play with the degree until the sigma of residuals is minimal.
# Typically deg of 4 to 7.


#################################### 04 FIT RESIDUAL DATA ###############################
#
method = "Kriging"
if "Chebyshev" in method:
    # Here I add new code
    # Define the grid size
    n_pts = 50j
    grid_x, grid_y = np.mgrid[1:dimx:n_pts,1:dimy:n_pts]

    ############ FIT ##########
    # we fit a 2D Chebyshev polynomial of degrees deg
    #
    # NB. The coefficients work on coordinates scaled to (-1, 1) in both axes
    #
    deg = (4, 4)
    zz_corr, coeffs, residuals, rank, s = fit_chebyshev(x_subset, y_subset, z_subset, z_subset_err, dimx, dimy, deg)
    inter_type = 'rbf'    # inter_type: 'cubic' or 'rbf' -- note that cubic uses 'linear' near the edges.

    # Define some strings for plotting    
    fit_image_title = f"{image_name}: cheb ({deg[0]},{deg[1]}) fit to residuals"
    fit_plot = image_path+"_cheb_"+f"{deg[0]}_{deg[1]}_"+inter_type+"_fit_01.png"
    coeff_file = image_path+"_cheb_"+f"{deg[0]}_{deg[1]}_"+inter_type+"_fit_01.txt"
    res_plot_str = image_path+"_cheb_"+f"{deg[0]}_{deg[1]}_"+inter_type+"_res"

    ############## USE POLYNOMIAL DIRECTLY ###########
    # we get explicit expressions for the polynomials
    # We do this to ensure that the expressions we provide, perhaps in headers,
    # have been validated by this notebook.
    #
    coefficients, cheb_poly_x_strdic, cheb_poly_y_strdic = chebyshev_polynomials(deg, coeffs, coeff_file, origfile, dimx=dimx, dimy=dimy)
    corrections = evaluate_2dpoly(x_subset, y_subset, dimx, dimy, deg, cheb_poly_x_strdic, cheb_poly_y_strdic, coefficients)
    z_corr = z_subset - corrections
    
    # we display the results of the fit as an image
    # NB: for Chebyshev we fit the whole data set as the
    #     method does not have the issue of exact interpolation
    #     at the input points, and have fewer degrees of freedom.
    #

    ################### EVALUATE POLYNOMIAL ON REGULAR GRID #######################
    #
    # chebyshev_2d_eval(x_pix, y_pix, coeff, deg)
    #
    if 'cubic' in inter_type:
        grid_z_cubic = griddata((x_subset, y_subset),
                       evaluate_2dpoly(x_subset, y_subset, dimx, dimy, deg, cheb_poly_x_strdic, cheb_poly_y_strdic, coefficients)
                       #chebyshev_2d_eval(x_subset, y_subset, dimx, dimy, coeffs, deg), 
                      (grid_x, grid_y), method='cubic')
        grid_z_nearest = griddata((x_subset, y_subset), 
                       #chebyshev_2d_eval(x_subset, y_subset, dimx, dimy, coeffs, deg),
                       evaluate_2dpoly(x_subset, y_subset, dimx, dimy, deg, cheb_poly_x_strdic, cheb_poly_y_strdic, coefficients)
                      (grid_x, grid_y), method='nearest')
        grid_z = np.where(np.isnan(grid_z_cubic), grid_z_nearest, grid_z_cubic)
    elif 'rbf' in inter_type:
        rbf = Rbf(x_subset, y_subset, chebyshev_2d_eval(x_subset, y_subset, dimx, dimy, coeffs, deg), function='cubic')
        grid_z = rbf(grid_x, grid_y)
        
    z_on_grid = grid_z.T
    
    print("z_on_grid.shape: {}".format(z_on_grid.shape))

    # Due to the low number of degrees of freedom for Chebyshev check points are all points
    x_check = x_subset
    y_check = y_subset
    z_check = z_subset
    err_type = "err_simple"

elif "Kriging" in method:                       # Not working yet - kernel dies
    # Perform Universal Kriging interpolation
    # This is a very expensive process. Data needs to be filtered out.
    # O(N^3+MxN^2) where N: sample data points, M: number of grid points
   
    logging.info(f"number of points for krigging: {len(x_subset)}")

    # set Kriging parameters
    maxlag = 8000
    nlags = 200
    selected_model = 'gaussian'        # exponential, gaussian, or spherical
    init_kriging = True                # Use a call to uk to determine parameters    
    use_empirical_variogram = True
    determine_empirical_variogram = True

    # Define some styrings for the plots
    fit_image_title = f"{image_name}: kriging {selected_model}\n nlags: {nlags} maxlag: {maxlag} interp. to residuals"
    fit_plot = image_path+"_krige_"+selected_model+"_fit_01.png"
    res_plot_str = image_path+'_krige_'+selected_model+"_res"

    # we determine best interpolation using all points
    plot_variogram = True
    UK = kriging_interpolation(x_subset, y_subset, z_subset, 
                    init_kriging = init_kriging, 
                    selected_model = selected_model, 
                    maxlag = maxlag, 
                    nlags = nlags, 
                    determine_empirical_variogram = determine_empirical_variogram, 
                    plot_variogram = plot_variogram,
                    use_empirical_variogram = determine_empirical_variogram,
                    image_path=image_path)

    # We do 100/A runs to determine kriging errors
    # k-fold cross-validation procedure using k% of points at a time for each determination
    # until all points have been used.
    #
    determine_errors = True
    use_kfold = False
    if determine_errors is True:
        if use_kfold:     # this is the proper way but it is slow. In a first pass set use_kfold to False
            k = 20         # should be small enough not to affect quality of estimation, but the smaller the costier
            x_check, y_check, z_check, z_corr = determine_kriging_errors(x_subset, y_subset, z_subset, 
                                         k=k,
                                         init_kriging=init_kriging, 
                                         selected_model=selected_model, 
                                         maxlag=maxlag, 
                                         nlags=nlags, 
                                         determine_empirical_variogram=determine_empirical_variogram, 
                                         plot_variogram=plot_variogram, 
                                         use_empirical_variogram=use_empirical_variogram)
            print("z_corr.shape: {}".format(z_corr.shape))
            err_type = f"err_kfold{k}"
        else:
            x_check = x_subset
            y_check = y_subset
            z_check = z_subset
            dpix = 1
            corr, _ = UK.execute('points', x_check+dpix, y_check+dpix)    # we offset a little otherwise corrections would be zero
            z_corr = z_check - corr
            err_type = "err_simple"
            


    # Interpolate onto the grid
    # Here I add new code
    # Define the grid size
    n_pts = 50
    gridx = np.linspace(1, dimx, n_pts)
    gridy = np.linspace(1, dimy, n_pts)

    # we get the values and the variances
    z_on_grid, ss = UK.execute('grid', gridx, gridy)

    # we create an image
    grid_z = z_on_grid
    grid_x, grid_y = np.meshgrid(gridx, gridy)
    print("z_on_grid.shape: {} --  grid_x.shape: {}".format(z_on_grid.shape, grid_x.shape))
    


logging.info("######################### DONE ######################")


In [ ]:
########################################## 05 PLOT APPROXIMATING SURFACE ###########################
#
fig, ax = plt.subplots(figsize=(12,12))
img = ax.imshow(z_on_grid, extent=(1, dimx, 1, dimy), origin='lower', cmap='viridis')
ax.set_title(fit_image_title, fontsize=24)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size="5%", pad=0.1)
cbar = plt.colorbar(img, cax=cax)
cbar.set_label('Residual [mag]', fontsize=14)
cbar.ax.tick_params(labelsize=14)

# Define contours and label them
contour_levels_neg = np.arange(-0.20, 0.0, 0.02)
contour_levels_pos = np.arange(0.02, 0.22, 0.02)

contour_neg = ax.contour(grid_x, grid_y, grid_z, cmap='viridis', levels = contour_levels_neg)
contour_pos = ax.contour(grid_x, grid_y, grid_z, cmap='plasma', levels = contour_levels_pos)
zero_contour = ax.contour(grid_x, grid_y, grid_z, colors='white', linestyles='--', linewidths=2, levels = [0])

ax.clabel(contour_neg, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(contour_pos, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(zero_contour, inline=True, fontsize=10, fmt='%1.3f')

# Plot the sample of points used to determine the 2D approximation
XY_subset = np.array(list(zip(x_subset, y_subset)))
ax.scatter(XY_subset[:,0], XY_subset[:,1], s=2, color='white')

# Plot the set of points to be used to check uncertainty
XY_check = np.array(list(zip(x_check, y_check)))
ax.scatter(XY_check[:,0], XY_check[:,1], s=30, color='white', marker='+')


#plt.colorbar(contour, label='Illumination correction')
ax.set_xlabel('RA in pixels', fontsize=16)
ax.set_ylabel('DEC in pixels', fontsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)


logging.info("Saving fit plot to {}".format(fit_plot))
plt.savefig(fit_plot)

plt.show()

In [ ]:
######################### PLOT RESIDUALS AND RMS ESTIMATE USING INDEPENDENT DATA POINTS ##################
#
#image_path = zpcal.image_path[:-5]
#image_name = image_name.split("/")[-1]

# In cases where edge oscilations are large  we might need to clip
# the max and minimum values.
min_corr = -0.20
max_corr = 0.25
#z_corr = np.clip(z_corr, min_corr, max_corr)


z_mean_u, z_median_u, z_std_u = sigma_clipped_stats(z_subset, sigma=3.0, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_u, 4), round(z_median_u, 4), round(z_std_u, 4)))

z_mean, z_median, z_std = sigma_clipped_stats(z_corr, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean, 4), round(z_median, 4), round(z_std, 4)))

# We save information into the headers of the coeff file if using Chebysheb
if "Chebyshev" in method:
    append_hashline(coeff_file, f"# uncorrected median: {round(z_median_u,4)}, mean: {round(z_mean_u,4)}, sigma: {round(z_std_u,4)}")
    append_hashline(coeff_file, f"# corrected median: {round(z_median,4)}, mean: {round(z_mean,4)}, sigma: {round(z_std,4)} factor: {round(z_std_u/z_std, 2)}")

fig = plt.figure(figsize=(12,5))
fig.suptitle(image_name+res_plot_str[0:-1], fontsize=16)
gs = GridSpec(1, 3, figure = fig)

# Residuals plot
ax1 = fig.add_subplot(gs[0, 0:2])
ax1.scatter(x_subset, z_subset, s=10, color='red', label="mean: {}   stdev: {}".format(round(z_mean_u, 3), round(z_std_u, 3)))
ax1.scatter(x_check, z_corr, color='black', s=5, marker='+', label='mean: {}  stdev: {}'.format(round(z_mean, 3), round(z_std, 3)))
ax1.axhline(z_mean, color='k', linewidth=1, linestyle='-')
ax1.axhline(z_mean+z_std, color='k', linewidth=1, linestyle='--')
ax1.axhline(z_mean-z_std, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.set_ylim(-0.25, 0.25)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

# Histogram plot
x_val = np.linspace(-0.25, 0.25, 100)
p = norm.pdf(x_val, z_mean, z_std)
ax2 = fig.add_subplot(gs[0, 2])

# We color the uncorrected 1-sigma interval
#ax2.axvline(z_mean_subset, color='k', linewidth=1, linestyle='-', label='mean: {}'.format(round(z_mean_subset, 3)))
ax2.axvspan(z_mean_subset-z_std_subset, z_mean_subset+z_std_subset, 
            color='gray', alpha=0.5,
            label='sigma: {}'.format(round(z_std_subset, 3)))

# We color the corrected 1-sigma interval
ax2.axvspan(z_mean-z_std, z_mean+z_std, 
            color='yellow', alpha=0.9,
            label='sigma: {}'.format(round(z_std, 3)))
ax2.axvline(z_mean+z_std, color='k', linewidth=1, linestyle='--', label='sigma: {}'.format(round(z_std, 3)))
ax2.axvline(z_mean-z_std, color='k', linewidth=1, linestyle='--')

ax2.hist(zpcal.z_masked, bins=30, density=True, stacked=True, color='green', alpha=1.0)
ax2.hist(z_corr, bins=30, density=True, stacked=True, color='blue', alpha=0.7)
plt.plot(x_val, p, 'k', linewidth=2)
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)
#ax2.set_ylim(0, 35)
ax2.text(0.80, 0.9, f"sigma x: {round(z_std_subset/z_std, 2)}", 
                      fontsize=11, 
                      transform=ax2.transAxes,
                      bbox=dict(facecolor='lightgray', alpha=0.5),
                      horizontalalignment='center')

plt.tight_layout()
logging.info("Saving residual plots to {}".format(res_plot_str+"_01.png"))
plt.savefig(res_plot_str+"_"+err_type+"_01.png")

plt.show()

In [ ]:
############################ EXTENSIVE SPATIAL PLOTS OF RESIDUALS ###########################
#
r = np.sqrt(x_subset**2+y_subset**2)
theta = np.degrees(np.arctan2(y_subset-dimy/2.0, x_subset-dimx/2.0))

r_check = np.sqrt(x_check**2+y_check**2)
theta_check = np.degrees(np.arctan2(y_check-dimy/2.0, x_check-dimx/2.0))

# fit a line to the radial dependence
from scipy import stats

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12,5))
fig.suptitle(image_name+res_plot_str[0:-1]+"\nResiduals of corrected and uncorrected magnitude differences")

slope, intercept, r_value, p_value, std_err = stats.linregress(r, z_subset)
ax1.scatter(r, z_subset, color='red', s=6, label="uncorrected mags")
xx = np.linspace(0, max(r), 100)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='red')
ax1.plot(xx, slope*xx+intercept, color='red')
ax1.set_xlabel("r [pix]")
ax1.set_ylabel("dm") 
ax1.set_ylim(-0.25, 0.25)
               
ax1.scatter(r_check, z_corr, color='black', s=2, label="corrected mags")
slope, intercept, r_value, p_value, std_err = stats.linregress(r_check, z_corr)
xx = np.linspace(0, 1.2, 10)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='black')
ax1.plot(xx, slope*xx+intercept, color='black')

ax1.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax1.legend()

ax2.scatter(theta, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax2.scatter(theta_check, z_corr, color='black', s=2, label="Residual of uncorrected mags")
ax2.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax2.set_xlabel("theta [deg]")
ax2.set_ylabel("dm")
ax2.set_ylim(-0.25, 0.25)

ax3.scatter(x_subset, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax3.scatter(x_check, z_corr, color='black', s=2, label="Residual of corrected mags")
ax3.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax3.set_xlabel("RA [pix]")
ax3.set_ylabel("dm")
ax3.set_ylim(-0.25, 0.25)

ax4.scatter(y_subset, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax4.scatter(y_check, z_corr, color='black', s=2, label="Residual of ucorrected mags")
ax4.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax4.set_xlabel("Dec [pix]")
ax4.set_ylabel("dm")
ax4.set_ylim(-0.25, 0.25)

plt.tight_layout()

logging.info("Saving residual plots to {}".format(res_plot_str+"_02.png"))
plt.savefig(res_plot_str+"_"+err_type+"_02.png")
               
plt.show()

In [ ]:
# We anlyze the bright saturated end to see if we can model it
# DOES NOT WORK. RESULT IS CLOSER TO STRAIGHT LINE UNTIL
# SAT MAG WHERE IT IS 0.0
#
from scipy.optimize import curve_fit
def sat_model(x, m, c, k, x0):
    return c + (m / k) * np.log(1 + np.exp(k * (x0 -  x)**2))

if "Chebyshev" in method:
    z_std = zpcal.mag_std
    z_frame = zpcal.mag_frame
    dm = z_std - z_frame

    x_pix = zpcal.x
    y_pix = zpcal.y

    z_corr = z_std - (z_frame + chebyshev_2d_eval(x_pix, y_pix, dimx, dimy, coeffs, deg))

    # Estimación inicial de los parámetros [m, c, k, x0]
    initial_guess = [0.1, 2.0, 1.0, 13]

    # Ajustar la función a los datos
    good_data = ~np.isnan(z_corr) & np.isfinite(z_corr) & ~np.isnan(z_std) & np.isfinite(z_std)
    params, covariance = curve_fit(sat_model, z_std[good_data], z_corr[good_data], p0=initial_guess)

    # Generar los datos ajustados
    x_data = np.linspace(8, 19, 100)
    y_fitted = sat_model(x_data, *params)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.scatter(z_std[good_data], z_corr[good_data], label='Datos', color='blue', s=3)
    ax.plot(x_data, y_fitted, label='Función ajustada', color='red')
    ax.set_ylim(-0.2, 0.2)
    ax.axhline(0.0, color='k', linewidth=1, linestyle='--')

    plt.show()

### 